# earthaccess plugins overview

earthaccess plugins reach services that are not part of `earthaccess` itself.
Each plugin is a small, independently installable package that appears under
`ea.plugins` once installed.

This notebook walks through the three reference plugins:

- **harmony** — subset/reproject/reformat with NASA Harmony (remote, resumable job)
- **pydap** — lazy OPeNDAP access as a `xarray.Dataset`
- **sliderule** — on-demand ICESat-2 / GEDI processing as a `GeoDataFrame`

## Setup

```bash
pip install earthaccess earthaccess-plugin-harmony earthaccess-plugin-pydap earthaccess-plugin-sliderule
```

All three plugins need Earthdata Login (SlideRule handles its own credentials).

In [ ]:
import earthaccess as ea

ea.login()  # strategy="netrc" by default; try strategy="interactive" if needed

In [ ]:
ea.list_plugins(details=True)

## Harmony

[Harmony](https://harmony.earthdata.nasa.gov/) is NASA's asynchronous,
cloud-based transformation service. `request()` returns immediately; `open()`
and `download()` block until the order finishes.

In [ ]:
atl06 = ea.search_data(
    short_name="ATL06",
    bounding_box=(-120, 40, -110, 45),
    temporal=("2020-01-01", "2020-02-01"),
)
len(atl06)

In [ ]:
job = ea.plugins.harmony.request(
    atl06,
    bbox=(-120, 40, -110, 45),
    variables=["/gt1l/land_ice_segments/h_li"],
)
job.status()

In [ ]:
dataset = ea.open(job)  # blocks until Harmony finishes
dataset

### Managing a Harmony order

Harmony jobs are remote and resumable. A large request waits in the preview
state (`JobStatus.PAUSED`); call `skip_preview()` to accept it. Pause, resume
and cancel are available too, and a job can be rebuilt from its id in another
process.

In [ ]:
# If the request is large, this returns PAUSED and open()/results() would raise
# JobPausedError. Accepting the preview resumes it:
job.status()
# job.skip_preview()   # accept a big request and run it
# job.pause(); job.resume()
# job.cancel()
# ea.plugins.harmony.get_job(job.id)   # resume from another process

## PyDAP

Inspect a granule with the native client, then request the variables you need
and open the OPeNDAP URLs that `ea.open(job)` returns.

In [ ]:
pyds = ea.plugins.pydap.request(modis, client="pydap").results()[0]
pyds.tree()

In [ ]:
import xarray as xr

session = ea.get_requests_https_session()
job = ea.plugins.pydap.request(modis, variables=["chlor_a"])
urls = ea.open(job)
ds = xr.open_dataset(urls[0], engine="pydap", session=session)
ds["chlor_a"].isel(rows=slice(0, 10)).mean().compute()

In [ ]:
# ea.download(job, "./data") serializes the constrained OPeNDAP data to NetCDF.

## SlideRule

[SlideRule](https://slideruleearth.io/) processes ICESat-2 and GEDI data in the
cloud. `request(...).results()` sends an API, parameters and a region of
interest and returns geolocated points as a `GeoDataFrame`. The searched
granules scope the request.

In [ ]:
ea.plugins.sliderule.configure(url="https://slideruleearth.io", organization="my-org")

In [ ]:
aoi = {
    "type": "Polygon",
    "coordinates": [[[-120, 40], [-110, 40], [-110, 45], [-120, 45], [-120, 40]]],
}
df = ea.plugins.sliderule.request(
    atl06,
    api="atl06p",
    parms={"pass_invalid": False},
    aoi=aoi,
).results()
df.head()

## Collection support

Passing results a plugin cannot handle fails early with a clear error, and you
can ask which plugins accept a dataset.

In [ ]:
atl06.list_services()

In [ ]:
ea.plugins.harmony.capabilities(atl06)  # Capabilities(ok, operations, formats)
ea.plugins.sliderule.capabilities("ATL06")  # by short name

In [ ]:
landsat = ea.search_data(
    short_name="LANDSAT_8_C1",
    temporal=("2023-06-01", "2023-06-02"),
)
try:
    ea.plugins.sliderule.request(landsat, api="atl06p").results()
except Exception as exc:
    print(type(exc).__name__, exc)

# A dataset you know the service accepts can bypass the check:
# ea.plugins.sliderule.request(results, api="atl06p", check=False).results()

## Learn more

- Plugin guides: [harmony](../plugins/harmony.md), [pydap](../plugins/pydap.md), [sliderule](../plugins/sliderule.md)
- Write your own: `cookiecutter gh:betolink/earthaccess-plugin-spec --directory template`